# **선형모델**


---
---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록  ───────────────────────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)



---



## **1. 선형모델 이해**
- **선형모델**은 머신러닝의 기초이자 많은 복잡한 모델의 출발점 해석 가능성이 중요한 상황에서 특히 유용함
- 선형 모델은 입력 특징(feature)의 선형 결합( 𝑤^⊤ 𝑥+𝑏)으로 예측하고  
- 분류는 이를 로지스틱/소프트맥스로 확률화하며,  
- 정규화(L1/L2) 로 과적합을 억제하는  
- 빠르고 해석 가능한 볼록 최적화 기반의 **기본선(Baseline)모델**

### 예제 : 간단 선형모델

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.datasets import make_regression, make_classification

# 회귀 데이터
X_reg, y_reg = make_regression(n_samples=100, n_features=5, random_state=42)

# 분류 데이터
X_clf, y_clf = make_classification(n_samples=100, n_features=5, random_state=42)

# 회귀 모델들
# model = LinearRegression()
# linear_reg =model.fit(X_reg, y_reg)
linear_reg = LinearRegression().fit(X_reg, y_reg)
ridge_reg = Ridge(alpha=1.0).fit(X_reg, y_reg)
lasso_reg = Lasso(alpha=1.0).fit(X_reg, y_reg)

# 분류 모델
logistic_clf = LogisticRegression().fit(X_clf, y_clf)

print("선형회귀 R²:", linear_reg.score(X_reg, y_reg))
print("릿지회귀(L2) R²:", ridge_reg.score(X_reg, y_reg))
print("라쏘회귀(L1) R²:", lasso_reg.score(X_reg, y_reg))
print("로지스틱회귀 정확도:", logistic_clf.score(X_clf, y_clf))

### 예제 : Tensorflow/Keras를 이용한 다층 신경망 회귀 모델

In [ ]:
import tensorflow as tf
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# 복잡한 데이터 생성
X, y = make_regression(n_samples=1000,
                       n_features=10,
                       noise=0.1,
                       random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("=== 다층 신경망 회귀 모델 ===")

# 다층 모델 구성
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(10,)),  # 은닉층 1
    tf.keras.layers.Dense(32, activation='relu'),                     # 은닉층 2
    tf.keras.layers.Dense(1)                                          # 출력층
])

# 컴파일 및 훈련
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.fit(X_train, y_train, epochs=100, validation_split=0.2, verbose=1)

# 평가
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"테스트 MSE: {test_loss:.2f}")
print(f"테스트 MAE: {test_mae:.2f}")

### **예제 : PyTorch 버전**
make_regression → Train/Test 분리 → 64-32-1 신경망 → Adam → MSE → 100 epochs → Test MSE/MAE 흐름을 동일하게 구성

In [ ]:
# ============================================================
# PyTorch 다층 신경망 회귀 모델
# TensorFlow/Keras 코드를 PyTorch로 변환
# ============================================================

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# 1. 난수 고정
# ------------------------------------------------------------
np.random.seed(42)
torch.manual_seed(42)


# ------------------------------------------------------------
# 2. 회귀 데이터 생성
# ------------------------------------------------------------
X, y = make_regression(
    n_samples=1000,
    n_features=10,
    noise=0.1,
    random_state=42
)
print("X shape:", X.shape)
print("y shape:", y.shape)


# ------------------------------------------------------------
# 3. Train / Test 데이터 분리
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ------------------------------------------------------------
# 4. Train 데이터에서 Validation 데이터 분리
#    Keras의 validation_split=0.2에 해당
# ------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)


# ------------------------------------------------------------
# 5. NumPy → PyTorch Tensor 변환
# ------------------------------------------------------------
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

# 출력 shape를 (N,) → (N, 1)로 변경
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_val   = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)
y_test  = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print("\nTrain:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)


# ------------------------------------------------------------
# 6. DataLoader 생성
#    Keras의 기본 batch_size=32와 동일하게 설정
# ------------------------------------------------------------
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)


# ------------------------------------------------------------
# 7. 다층 신경망 모델 정의
# ------------------------------------------------------------
class RegressionModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            # 입력 feature 10개 → 은닉노드 64개
            nn.Linear(10, 64),
            nn.ReLU(),

            # 은닉노드 64개 → 32개
            nn.Linear(64, 32),
            nn.ReLU(),

            # 출력값 1개
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


model = RegressionModel()

print("\n=== 다층 신경망 회귀 모델 ===")
print(model)


# ------------------------------------------------------------
# 8. Loss와 Optimizer 설정
# ------------------------------------------------------------
# TensorFlow의 loss='mse'
criterion = nn.MSELoss()

# TensorFlow의 optimizer='adam'
optimizer = torch.optim.Adam(
    model.parameters()
)


# ------------------------------------------------------------
# 9. 모델 학습
# ------------------------------------------------------------
epochs = 100
for epoch in range(epochs):
    # -----------------------
    # Training
    # -----------------------
    model.train()

    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        # ① 예측
        pred = model(X_batch)

        # ② Loss 계산
        loss = criterion(pred, y_batch)

        # ③ 이전 gradient 초기화
        optimizer.zero_grad()

        # ④ Backpropagation
        loss.backward()

        # ⑤ Weight 업데이트
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)


    train_loss /= len(train_loader.dataset)


    # -----------------------
    # Validation
    # -----------------------
    model.eval()

    with torch.no_grad():
        val_pred = model(X_val)
        val_loss = criterion(
            val_pred,
            y_val
        ).item()


    # 10 epoch마다 출력
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch+1:3d}/{epochs}] "
            f"Train MSE: {train_loss:8.4f} "
            f"Val MSE: {val_loss:8.4f}"
        )


# ------------------------------------------------------------
# 10. Test 데이터 평가
# ------------------------------------------------------------
model.eval()

with torch.no_grad():
    y_pred = model(X_test)

    # MSE
    test_mse = criterion(
        y_pred,
        y_test
    ).item()

    # MAE
    test_mae = torch.mean(
        torch.abs(y_pred - y_test)
    ).item()


print("\n=== 테스트 결과 ===")

print(f"테스트 MSE: {test_mse:.2f}")
print(f"테스트 MAE: {test_mae:.2f}")



---



## **2. 선형회귀**

- 데이터의 추세를 그래프로 표현하여 이후의 패턴을 예측하는 데 활용하는 통계 분석 방법론 --> **회귀 분석**
- label(Y)과 feature(X)의 관계를 잘 설명하는 모델을 만드는 것 --> **머신러닝**
- 최적의 직선을 기반으로 예측하는 알고리즘 --> **선형 회귀(Linear Regression)**

- $𝑦(𝑤, 𝑥)=𝑤_0+ 𝑤_1 𝑥_1  +...+ 𝑤_𝑝 𝑥_𝑝$   
    - $ 𝒙 = 독립변수,   w = 가중치,   y = 종속변수$

### **예제 : 기본 선형회귀**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# 1. 데이터 생성
X, y = make_regression(n_samples=100, n_features=1, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. 모델 생성 및 훈련
model = LinearRegression()
model.fit(X_train, y_train)

# 3. 예측
y_pred = model.predict(X_test)

# 4. 성능 평가
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R² Score: {r2:.3f}")
print(f"MSE: {mse:.3f}")
print(f"모델 계수: {model.coef_[0]:.3f}")
print(f"절편: {model.intercept_:.3f}")

### **예제 : Tensorflow(딥러닝)를 이용한 선형회귀**

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.rcParams["font.family"] = "NanumBarunGothic"

print("TensorFlow 버전:", tf.__version__)

# 1. 데이터 준비
print("=== 데이터 생성 ===")
X, y = make_regression(n_samples=1000, n_features=1, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 데이터 정규화 (선택사항이지만 권장)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"훈련 데이터 크기: {X_train.shape}")
print(f"테스트 데이터 크기: {X_test.shape}")

# 2. 모델 구성
print("\n=== 모델 구성 ===")
model = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(1,))  # 입력 1개, 출력 1개
])

# 3. 모델 컴파일
model.compile(
    optimizer='adam',           # 최적화 알고리즘
    loss='mean_squared_error',  # 손실 함수
    metrics=['mae']             # 평가 지표
)

# 모델 구조 확인
model.summary()

# 4. 모델 훈련
print("\n=== 모델 훈련 ===")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,              # 훈련 반복 횟수
    batch_size=32,           # 배치 크기
    validation_split=0.2,    # 검증 데이터 비율
    verbose=1                # 훈련 과정 출력
)

# 5. 모델 평가
print("\n=== 모델 평가 ===")
test_loss, test_mae = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"테스트 손실 (MSE): {test_loss:.2f}")
print(f"테스트 MAE: {test_mae:.2f}")

# 6. 예측
y_pred = model.predict(X_test_scaled)

# R² Score 계산
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred)
print(f"R² Score: {r2:.3f}")

# 7. 결과 시각화
plt.figure(figsize=(12, 4))

# 훈련 과정
plt.subplot(1, 3, 1)
plt.plot(history.history['loss'], label='훈련 손실')
plt.plot(history.history['val_loss'], label='검증 손실')
plt.title('훈련 과정')
plt.xlabel('Epoch')
plt.ylabel('손실')
plt.legend()

# 예측 결과
plt.subplot(1, 3, 2)
plt.scatter(X_test, y_test, alpha=0.6, label='실제값')
plt.scatter(X_test, y_pred, alpha=0.6, label='예측값')
plt.title('예측 결과')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()

# 실제 vs 예측
plt.subplot(1, 3, 3)
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('실제 vs 예측')
plt.xlabel('실제값')
plt.ylabel('예측값')

plt.tight_layout()
plt.show()

### **[실습] 잡음이 있는 y = 4x + 7 단순 선형회귀**

In [ ]:
# ============================================================
# 단순 선형회귀 실습
# y = 4x + 7 + noise
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score


# ------------------------------------------------------------
# 1. 데이터 생성
# ------------------------------------------------------------
# 난수 생성기
rng = np.random.default_rng(42)

# 입력 데이터 X
# 0~10 사이의 난수 100개
# shape = (100, 1)
X = rng.uniform(0, 10, size=(100, 1))

# 실제 관계:
# y = 4x + 7 + noise
#
# 실제 가중치 w = 4
# 실제 절편 b = 7
# noise 표준편차 = 4
y = 4 * X[:, 0] + 7 + rng.normal(
    loc=0,
    scale=4,
    size=100
)


# ------------------------------------------------------------
# 2. Train / Test 데이터 분리
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                    test_size=0.2, random_state=42 )
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)


# ------------------------------------------------------------
# 3. 선형회귀 모델 학습
# ------------------------------------------------------------
model = LinearRegression()
model.fit(X_train, y_train)


# ------------------------------------------------------------
# 4. Test 데이터 예측
# ------------------------------------------------------------
y_pred = model.predict(X_test)


# ------------------------------------------------------------
# 5. 학습된 회귀식 확인
# ------------------------------------------------------------
w = model.coef_[0]
b = model.intercept_

print("\n학습된 모델")
print(f"w = {w:.3f}")
print(f"b = {b:.3f}")

print(f"\n회귀식: y = {w:.3f}x + {b:.3f}")


# ------------------------------------------------------------
# 6. 모델 평가
# ------------------------------------------------------------
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n모델 평가")
print(f"RMSE = {rmse:.3f}")
print(f"R²   = {r2:.3f}")


# ------------------------------------------------------------
# 7. 결과 시각화
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))

# 학습 데이터
plt.scatter(
    X_train[:, 0],
    y_train,
    s=20,
    label="Train"
)

# 테스트 데이터
plt.scatter(
    X_test[:, 0],
    y_test,
    marker="^",
    s=40,
    label="Test"
)

# 회귀선용 X 값 생성
xs = np.linspace(0, 10, 100).reshape(-1, 1)

# 학습된 회귀선
plt.plot(
    xs[:, 0],
    model.predict(xs),
    linewidth=2,
    label="Regression Line"
)

plt.xlabel("X")
plt.ylabel("y")
plt.title("Linear Regression")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

### **[실습] 사례기반 선형회귀 모델링**

#### **1) 데이터 전처리하기**

In [ ]:
# 판다스 라이브러리 불러오기
import pandas as pd

In [ ]:
# 데이터 불러오기
df = pd.read_csv("국민건강보험공단_건강검진정보_20211229.CSV", encoding='cp949')

In [ ]:
# pandas display 옵션 조정을 통해 View 범위 확장하기
pd.set_option('display.max_columns', None) # display 옵션을 통한 전체 열 확장

In [ ]:
# 데이터 확인하기 (상위 5개)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# 가설을 참고하여 데이터 일부 삭제하기
# 시력, 청력, 치아 관련 칼럼은 관계없다는 가정으로 열 제거하기
df.drop(["치아우식증유무", '치석','시력(좌)', '시력(우)', '청력(좌)',  '청력(우)', '구강검진 수검여부'], axis=1, inplace=True)

In [ ]:
# 기준년도 칼럼 확인하기 (모두 '2020' 동일 값이므로 '기준년도' 칼럼 삭제)
print("기준년도 칼럼 확인")
df.기준년도.value_counts()

In [ ]:
# 가입자 일련번호 칼럼 확인하기 (모두 unique한 값으로 확인되어 '가입자 일련번호' 컬럼 삭제)
print("가입자 일련번호 칼럼 확인")
df["가입자 일련번호"].value_counts()

In [ ]:
# 성별코드 칼럼 확인하기
print("성별코드 칼럼 확인")
df.성별코드.value_counts()

In [ ]:
# 불필요한 데이터 삭제하기
df.drop(["기준년도", '가입자 일련번호', '데이터 공개일자', '성별코드', '시도코드'], axis=1, inplace=True)

In [ ]:
# 별도의 test 데이터 추출하기
test = df[df['LDL 콜레스테롤'].isnull()]

In [ ]:
# NaN 데이터 행 단위로 삭제하기
train = df.dropna(axis=0)

In [ ]:
# 학습 데이터 확인하기
print("학습 데이터 확인")
train.head(1)

In [ ]:
# 정답 데이터 생성하기
y = train['LDL 콜레스테롤']

In [ ]:
# 학습 데이터 생성하기
x = train.drop('LDL 콜레스테롤', axis=1)

In [ ]:
# validation set 추출을 위한 train_test_split 라이브러리 불러오기
from sklearn.model_selection import train_test_split

In [ ]:
# scikit learn 예시 코드 비율 대로 불러오기
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.33, random_state=42)

In [ ]:
# 학습/검증 데이터 확인하기
print("학습/검증 데이터 확인")
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

#### **2) 선형회귀 학습 및 추론하기**

In [ ]:
# 1. 라이브러리 불러오기
from sklearn.linear_model import LinearRegression
# 2. 모델 생성하기
reg = LinearRegression()
# 3. 학습하기
reg.fit(X_train,y_train)
# 4. 예측하기
reg.predict(X_test)

In [ ]:
# 기울기와 절편 확인하기
print(f'''기울기 및 절편 확인
기울기확인 coef = {reg.coef_}
절편확인 intercept = {reg.intercept_}''')

In [ ]:
# 각각 데이터에 대해 가중치(or 회귀계수) 확인하기
print("전체 컬럼에 대해서 가중치 확인")
print('-'*30)
for index ,columns in enumerate(X_train.columns):
    print(f"{columns}(x{index}) = {reg.coef_[index]}")

#### **3) 예측을 통한 최종 검증하기**

In [ ]:
# 예측하기
y_pred = reg.predict(X_test)

In [ ]:
# 결과 검증을 위해 MSE 라이브러리 불러오기
# 최종적으로는 RMSE를 사용하기
from sklearn.metrics import mean_squared_error

In [ ]:
# mse 라이브러리 에서 RMSE 는 squared 옵션을 False로 설정하기
rmse = mean_squared_error(y_test, y_pred)

In [ ]:
# 주요 Feature 삭제 전 rmse 확인하기
print(f'''주요 Feature 삭제 전 rmse = {round(rmse,3)}''')

#### **4) 가중치의 의미 확인하기**

In [ ]:
# 주요 Feature 삭제를 위해 칼럼명 재확인하기
train.columns

In [ ]:
# 주요 Feature 삭제하기
x = x.drop(['총 콜레스테롤', '트리글리세라이드', 'HDL 콜레스테롤'],axis = 1)

In [ ]:
# scikit learn 예시 코드 비율 대로 불러오기
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.33, random_state=42)

#### **5) 선형회귀 모델 재학습하기**

In [ ]:
# 선형회귀 재생성 및 학습하기
lr = LinearRegression()
lr.fit(X_train,y_train)

In [ ]:
# 각각 데이터에 대해 가중치(or 회귀계수) 확인하기
print(f'''coef
{lr.coef_}
intercept
{lr.intercept_}''')

#### **6) 가중치 비교하기**

In [ ]:
# 검증 데이터로 예측하기
y_pred = lr.predict(X_test)

In [ ]:
# 각각 데이터에 대해 가중치(또는 회귀계수) 확인하기
print("가중치 확인")
for index ,columns in enumerate(X_train.columns):
    print(f"{columns} = {reg.coef_[index]}")

In [ ]:
# 주요 Feature 삭제 후 RMSE 확인하기
rmse_2 = mean_squared_error(y_test, y_pred)

In [ ]:
# 가중치가 큰 특성을 삭제 했을 때 영향을 많이 받는지 확인하기
print(f'''주요 Feature 삭제 후 rmse = {round(rmse_2,3)}''')



---



### **[실습] 전처리 + 회귀 Pipeline → 평가 → Residual**

In [ ]:
# ============================================================
# 학생 성적 데이터 회귀 실습
# 전처리 + Linear Regression + Dummy Regression + 성능평가
# ============================================================

# 1. 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)


# ------------------------------------------------------------
# 2. CSV 파일 업로드
# ------------------------------------------------------------
uploaded = files.upload()

# student_scores.csv 파일 읽기
stu = pd.read_csv("student_scores.csv")

print("데이터 크기:", stu.shape)
print("\n[처음 5행]")
display(stu.head())

print("\n[데이터 정보]")
stu.info()


# ------------------------------------------------------------
# 3. Target(final_score)이 결측인 행 제거
# ------------------------------------------------------------
# 입력 feature의 결측치는 아래 SimpleImputer가 처리할 수 있지만
# 정답 y가 결측이면 모델 학습이 불가능하므로 제거합니다.

stu = stu.dropna(subset=["final_score"]).copy()

print("\nTarget 결측 제거 후 데이터 크기:", stu.shape)


# ------------------------------------------------------------
# 4. X, y 분리
# ------------------------------------------------------------
Xs = stu.drop(columns=["final_score"])
ys = stu["final_score"]

print("\nX shape:", Xs.shape)
print("y shape:", ys.shape)


# ------------------------------------------------------------
# 5. 수치형 / 범주형 feature 자동 분류
# ------------------------------------------------------------
num = Xs.select_dtypes(include=["number"]).columns.tolist()
cat = Xs.select_dtypes(exclude=["number"]).columns.tolist()

print("\n수치형 feature:", num)
print("범주형 feature:", cat)


# ------------------------------------------------------------
# 6. Train / Test 분리
# 회귀 문제이므로 stratify를 사용하지 않습니다.
# ------------------------------------------------------------
Xtr, Xte, ytr, yte = train_test_split(
    Xs,
    ys,
    test_size=0.2,
    random_state=42
)

print("\nTrain:", Xtr.shape)
print("Test :", Xte.shape)


# ------------------------------------------------------------
# 7. 전처리 Pipeline
# ------------------------------------------------------------

# 수치형:
# 결측치 → 중앙값
# Scaling → StandardScaler

num_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

# 범주형:
# 결측치 → 최빈값
# 범주 → One-Hot Encoding

cat_pipe = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
)

# 수치형과 범주형 처리 결합
pre_reg = ColumnTransformer([
    ("num", num_pipe, num),
    ("cat", cat_pipe, cat)
])


# ------------------------------------------------------------
# 8. Linear Regression 모델
# ------------------------------------------------------------
reg = make_pipeline(
    pre_reg,
    LinearRegression()
)

reg.fit(Xtr, ytr)


# ------------------------------------------------------------
# 9. Dummy Regression
# ------------------------------------------------------------
# 학습 데이터 final_score 평균만 예측하는 아주 단순한 모델
# → Linear Regression 성능 비교용 baseline

dum = DummyRegressor(strategy="mean")
dum.fit(Xtr, ytr)


# ------------------------------------------------------------
# 10. 모델 성능 평가
# ------------------------------------------------------------
print("\n========== 모델 성능 ==========")

results = []

for name, model in [
    ("Dummy", dum),
    ("Linear", reg)
]:

    pred = model.predict(Xte)

    mae = mean_absolute_error(yte, pred)
    rmse = root_mean_squared_error(yte, pred)
    r2 = r2_score(yte, pred)

    results.append([name, mae, rmse, r2])


result_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "RMSE", "R2"]
)

display(result_df.round(3))


# ------------------------------------------------------------
# 11. Linear Regression의 회귀계수 확인
# ------------------------------------------------------------
# 전처리 이후 feature 이름 가져오기

feature_names = (
    reg.named_steps["columntransformer"]
       .get_feature_names_out()
)

# Linear Regression 계수
coef = reg.named_steps["linearregression"].coef_

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coef
})

# 절댓값이 큰 순서
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)

print("\n========== Feature별 회귀계수 ==========")

display(
    coef_df[["feature", "coefficient"]]
    .round(3)
)


# ------------------------------------------------------------
# 12. 실제값과 예측값 비교
# ------------------------------------------------------------

pred = reg.predict(Xte)

compare_df = pd.DataFrame({
    "Actual": yte.values,
    "Predicted": pred,
    "Error": yte.values - pred
})

print("\n========== 실제값 vs 예측값 ==========")
display(compare_df.head(10).round(2))


# ------------------------------------------------------------
# 13. 잔차(Residual) 그래프
# ------------------------------------------------------------
resid = yte - pred

plt.figure(figsize=(8, 5))

plt.scatter(
    pred,
    resid,
    alpha=0.7
)

plt.axhline(
    0,
    color="red",
    linestyle="--"
)

plt.xlabel("Predicted final_score")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")

plt.show()

### **[실습] Linear vs Ridge vs Lasso 비교**

In [ ]:
# ============================================================
# Linear Regression vs Ridge vs Lasso
# 교차검증(Cross Validation) + alpha 최적화
# ============================================================
import numpy as np
import pandas as pd

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    RidgeCV,
    LassoCV
)

from sklearn.model_selection import (
    KFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error


# ------------------------------------------------------------
# 1. K-Fold Cross Validation 설정
# ------------------------------------------------------------
# Train 데이터를 5개 fold로 나누어
# 4개로 학습하고 1개로 검증하는 과정을 5번 반복
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ------------------------------------------------------------
# 2. Linear / Ridge / Lasso 모델 정의
# ------------------------------------------------------------
models = {
    "Linear": LinearRegression(),

    # L2 정규화
    "Ridge": Ridge(
        alpha=1.0
    ),

    # L1 정규화
    "Lasso": Lasso(
        alpha=0.5,
        max_iter=10000
    )
}


# ------------------------------------------------------------
# 3. Cross Validation으로 모델 비교
# ------------------------------------------------------------
print("========== 5-Fold Cross Validation ==========\n")

results = []

for name, model in models.items():

    # 전처리 + 모델
    pipe = make_pipeline(
        pre_reg,
        model
    )

    # RMSE 기준 교차검증
    scores = cross_val_score(
        pipe,
        Xtr,
        ytr,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )

    # sklearn은 오차 지표에 음수를 붙여 반환하므로
    # 다시 양수로 변환
    cv_rmse = -scores.mean()

    # 전체 Train 데이터로 다시 학습
    pipe.fit(Xtr, ytr)

    # 거의 0이 된 계수 개수
    coef = pipe[-1].coef_

    zeros = int(
        (np.abs(coef) < 1e-6).sum()
    )

    results.append([
        name,
        cv_rmse,
        zeros
    ])

    print(
        f"{name:<7} "
        f"CV-RMSE = {cv_rmse:.3f}   "
        f"0이 된 계수 = {zeros}"
    )


# 결과를 표로 출력
result_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "CV-RMSE",
        "Zero coefficients"
    ]
)

print("\n========== 모델 비교 ==========")

display(
    result_df.round(3)
)


# ============================================================
# 4. RidgeCV
# ============================================================
# alpha를 사람이 정하지 않고
# Cross Validation으로 자동 선택
#
# 0.001 ~ 1000 범위
# ============================================================
alphas = np.logspace(
    -3,
    3,
    13
)

print("\nRidge alpha 후보:")
print(alphas)


ridge_cv = make_pipeline(

    pre_reg,

    RidgeCV(
        alphas=alphas,
        cv=kf,
        scoring="neg_root_mean_squared_error"
    )
)


ridge_cv.fit(
    Xtr,
    ytr
)


# 선택된 alpha
best_ridge_alpha = ridge_cv[-1].alpha_


# Test 데이터 예측
ridge_pred = ridge_cv.predict(
    Xte
)


# Test RMSE
ridge_rmse = root_mean_squared_error(
    yte,
    ridge_pred
)


print(
    "\n========== RidgeCV =========="
)

print(
    "Best alpha:",
    best_ridge_alpha
)

print(
    "Test RMSE:",
    round(ridge_rmse, 3)
)


# ============================================================
# 5. LassoCV
# ============================================================
# Lasso의 alpha도 CV로 자동 선택
# ============================================================
lasso_cv = make_pipeline(

    pre_reg,

    LassoCV(
        cv=kf,
        random_state=42,
        max_iter=10000
    )
)


lasso_cv.fit(
    Xtr,
    ytr
)


# 선택된 alpha
best_lasso_alpha = lasso_cv[-1].alpha_


# Test 예측
lasso_pred = lasso_cv.predict(
    Xte
)


# Test RMSE
lasso_rmse = root_mean_squared_error(
    yte,
    lasso_pred
)


print(
    "\n========== LassoCV =========="
)

print(
    "Best alpha:",
    round(best_lasso_alpha, 5)
)

print(
    "Test RMSE:",
    round(lasso_rmse, 3)
)


# Lasso에서 0으로 제거된 feature 확인
lasso_coef = lasso_cv[-1].coef_

lasso_zero_count = int(
    (np.abs(lasso_coef) < 1e-6).sum()
)

print(
    "0이 된 계수:",
    lasso_zero_count
)


# ============================================================
# 6. GridSearchCV를 이용한 Ridge alpha 탐색
# ============================================================

param_grid = {

    "ridge__alpha": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}


gs = GridSearchCV(

    estimator=make_pipeline(
        pre_reg,
        Ridge()
    ),

    param_grid=param_grid,

    cv=kf,

    scoring="neg_root_mean_squared_error",

    # 여러 CPU core 사용
    n_jobs=-1
)


# ------------------------------------------------------------
# 7. scikit-learn 1.9 이상이면 ProgressBar 사용
# ------------------------------------------------------------
try:

    from sklearn.callback import ProgressBar

    print(
        "\nscikit-learn callback ProgressBar 사용"
    )

    gs.set_callbacks(
        ProgressBar()
    )

except (ImportError, AttributeError):

    print(
        "\n현재 scikit-learn 버전에서는 "
        "ProgressBar callback을 지원하지 않습니다."
    )

    print(
        "GridSearchCV는 ProgressBar 없이 정상 실행됩니다."
    )


# Grid Search 실행
gs.fit(
    Xtr,
    ytr
)


# ============================================================
# 8. GridSearchCV 결과
# ============================================================

print(
    "\n========== GridSearchCV =========="
)

print(
    "Best parameters:",
    gs.best_params_
)

print(
    "Best CV-RMSE:",
    round(
        -gs.best_score_,
        3
    )
)


# Test 성능
gs_pred = gs.predict(
    Xte
)

gs_rmse = root_mean_squared_error(
    yte,
    gs_pred
)

print(
    "Test RMSE:",
    round(
        gs_rmse,
        3
    )
)

### **[실습] 선형회귀로 LLM API 비용 예측**

In [ ]:
# ============================================================
# Gemini Token 수와 글자 수의 관계 분석
# - Gemini API의 실제 tokenizer로 토큰 수 측정
# - Linear Regression으로 글자 수 → 토큰 수 관계 추정
# - 100만 글자의 예상 토큰 수와 비용 계산
# ============================================================

# 1. 필요한 라이브러리 설치
!pip install -q -U google-genai scikit-learn pandas matplotlib

In [ ]:
# ============================================================
# Gemini Token 수 예측 실습
# ============================================================

!pip install -q -U google-genai scikit-learn pandas matplotlib

from google import genai
from google.colab import userdata

from sklearn.linear_model import LinearRegression

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Gemini API
# ------------------------------------------------------------

api_key = userdata.get("GEMINI_API_KEY")

if api_key is None:
    raise ValueError(
        "Colab Secrets에 GEMINI_API_KEY를 등록해주세요."
    )

client = genai.Client(
    api_key=api_key
)

MODEL = "gemini-3.5-flash"


# ------------------------------------------------------------
# 테스트 데이터
# ------------------------------------------------------------

texts = [
    "안녕하세요",

    "기계학습 프로그래밍 수업입니다",

    "트랜스포머 아키텍처는 어텐션 메커니즘을 기반으로 한다",

    "인공지능 " * 30,

    "머신러닝과 딥러닝의 차이를 설명하시오",

    "Machine learning finds rules from data.",

    "Attention is all you need."
]


df = pd.DataFrame({
    "text": texts
})


# ------------------------------------------------------------
# 글자 수
# ------------------------------------------------------------

df["글자수"] = (
    df["text"]
    .str.len()
)


# ------------------------------------------------------------
# Gemini 실제 token 수
# ------------------------------------------------------------

df["토큰수"] = [

    client.models.count_tokens(
        model=MODEL,
        contents=text
    ).total_tokens

    for text in texts
]


print("===== Token 측정 =====")

display(df)


# ------------------------------------------------------------
# Linear Regression
# ------------------------------------------------------------

X = df[["글자수"]]

y = df["토큰수"]


model = LinearRegression()

model.fit(
    X,
    y
)


print(
    "\n===== Linear Regression ====="
)

print(
    f"토큰수 ≈ "
    f"{model.coef_[0]:.3f} × 글자수 "
    f"+ {model.intercept_:.2f}"
)

print(
    f"R² = "
    f"{model.score(X, y):.3f}"
)


# ------------------------------------------------------------
# 회귀선
# ------------------------------------------------------------

plt.figure(
    figsize=(8, 5)
)


plt.scatter(
    df["글자수"],
    df["토큰수"],
    s=80,
    label="Actual"
)


x_line = np.linspace(
    df["글자수"].min(),
    df["글자수"].max(),
    100
)


X_line = pd.DataFrame({
    "글자수": x_line
})


y_line = model.predict(
    X_line
)


plt.plot(
    x_line,
    y_line,
    label="Linear Regression"
)


plt.xlabel(
    "Number of Characters"
)

plt.ylabel(
    "Number of Tokens"
)

plt.title(
    f"Characters vs Tokens ({MODEL})"
)

plt.legend()

plt.show()


# ------------------------------------------------------------
# 100만 글자 예측
# ------------------------------------------------------------

characters = 1_000_000


tokens = model.predict(

    pd.DataFrame({
        "글자수": [characters]
    })

)[0]


# ------------------------------------------------------------
# 비용
# ------------------------------------------------------------

# 현재 공식 Gemini 3.8 Flash 소개 가격:
# 입력 100만 토큰당 $0.75
# 실제 수업에서는 당일 공식 가격 확인 권장

PRICE_PER_1M = 0.75


cost = (
    tokens
    / 1_000_000
    * PRICE_PER_1M
)


print(
    "\n===== 100만 글자 예측 ====="
)

print(
    f"글자 수       : "
    f"{characters:,}"
)

print(
    f"예상 토큰 수  : "
    f"{tokens:,.0f}"
)

print(
    f"예상 입력 비용: "
    f"${cost:.2f}"
)